# 10장. 체크포인트

In [1]:
# [준비 셀] 모든 노트북의 첫 코드 셀입니다. 항상 먼저 실행하세요.
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[1]          # docs/manual → 프로젝트 루트
assert (ROOT / "src/bearing_kd").exists(), "docs/manual 폴더에서 노트북을 열어야 합니다"
CODE = ROOT / "docs/manual/code"                 # 챕터별 완성 스크립트 위치
sys.path.insert(0, str(CODE))                    # 스크립트의 함수를 import 해서 쓰기 위해
print("프로젝트 루트 확인 완료:", ROOT.name)

프로젝트 루트 확인 완료: vib-onnx


**체크포인트(checkpoint)**는 학습한 모델을 파일로 저장한 것입니다. 학습은 몇 시간이 걸리지만, 체크포인트가 있으면 몇 초 만에 같은 모델을 다시 불러올 수 있습니다.

PyTorch 체크포인트(`.pt`)는 **파이썬 딕셔너리**를 `torch.save`로 저장한 파일입니다. 무엇을 넣을지는 우리가 정합니다.

In [2]:
import os, warnings
os.environ["HF_HUB_OFFLINE"] = "1"                 # 인터넷 없이 로컬 모델 폴더만 사용
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"    # Mac: MPS가 모르는 연산은 CPU로
warnings.filterwarnings("ignore")
import torch
from ch09_train_teacher import pick_device
DEVICE = pick_device()
print("장치:", DEVICE)

장치: mps


In [3]:
import ch10_checkpoint as ch10
ckpt = ch10.describe(ROOT / "outputs/manual/teacher/teacher_lora_best.pt")

outputs/manual/teacher/teacher_lora_best.pt  (457.6 MB)
  epoch                  3
  model_state_dict       dict, 항목 164개, 숫자 113,771,906개 (455.1 MB)
  optimizer_state_dict   dict, 항목 2개
  validation_accuracy    1.0
  lora_config            dict, 항목 6개
  training_config        dict, 항목 9개


| 키 | 내용 | 왜 저장하나 |
|---|---|---|
| `model_state_dict` | 모델의 모든 숫자 (이름 → 텐서) | 모델 복원 |
| `optimizer_state_dict` | AdamW가 기억하는 기울기 통계 | 학습을 이어서 할 때 |
| `epoch`, `validation_accuracy` | 언제, 얼마나 잘했나 | 기록 |
| `lora_config` | LoRA 설정 (r, alpha, 대상) | 같은 구조를 만들어야 불러올 수 있음 |
| `training_config` | learning rate, seed, 데이터 지문 등 | 재현 |

## 10.1 state_dict 들여다보기

`state_dict`는 `"층 이름": 텐서` 딕셔너리입니다.

In [4]:
state = ckpt["model_state_dict"]
for k in list(state)[:4] + [k for k in state if "lora" in k][:2] + [k for k in state if k.startswith("head")]:
    print(f"{k:62s} {tuple(state[k].shape)}")
print("...\n전체 항목", len(state), "개")

patch_embedding.mask_embedding                                 (768,)
patch_embedding.value_embedding.weight                         (768, 8)
patch_embedding.position_embedding.pe                          (1, 5000, 768)
encoder.embed_tokens.weight                                    (32128, 768)
encoder.block.0.layer.0.SelfAttention.q.lora_A.default.weight  (8, 768)
encoder.block.0.layer.0.SelfAttention.q.lora_B.default.weight  (768, 8)
head.linear.weight                                             (2, 768)
head.linear.bias                                               (2,)
...
전체 항목 164 개


## 10.2 LoRA가 실제로 학습됐는지 확인하기

LoRA B는 0으로 시작하므로, 학습이 됐다면 0이 아닌 값이 있어야 합니다. 8장의 버그를 체크포인트만 보고도 잡아낼 수 있습니다.

In [5]:
print("매뉴얼로 학습한 Teacher:", ch10.lora_b_report(state))
old = ROOT / "outputs/checkpoints/teacher_lora_best.pt"
if old.exists():
    print("프로젝트의 옛날 Teacher:", ch10.lora_b_report(ch10.load_checkpoint(old)["model_state_dict"]), "← 버그 때문에 전부 0")

매뉴얼로 학습한 Teacher: (24, 24)


프로젝트의 옛날 Teacher: (0, 24) ← 버그 때문에 전부 0


## 10.3 안전하게 불러오기: `weights_only=True`

`.pt` 파일은 파이썬 객체를 저장할 수 있어서, 악의적으로 만든 파일은 **불러오는 순간 코드를 실행**할 수 있습니다. `torch.load(..., weights_only=True)`는 텐서와 기본 자료형만 읽어서 이를 막습니다. 남이 만든 체크포인트는 항상 이렇게 여세요.

## 10.4 strict 로딩: 조용한 실수 막기

`model.load_state_dict(state, strict=True)`는 모델의 층 이름과 파일의 이름이 **하나라도 다르면 에러**를 냅니다. `strict=False`는 맞는 것만 넣고 넘어가서, LoRA 설정이 다르거나 이름이 바뀌면 **학습 안 된 모델이 조용히 만들어집니다.**

In [6]:
teacher = ch10.load_full(ROOT / "models/MOMENT-1-base", ROOT / "outputs/manual/teacher/teacher_lora_best.pt")
print("strict=True 로딩 성공")

import ch07_moment as ch07
plain = ch07.load_moment(ROOT / "models/MOMENT-1-base")      # LoRA를 안 붙인 모델에 넣으면?
try:
    plain.load_state_dict(state, strict=True)
except RuntimeError as e:
    print("구조가 다르면 에러:", str(e).splitlines()[0][:90], "...")

Loading weights from local directory


strict=True 로딩 성공


Loading weights from local directory
구조가 다르면 에러: Error(s) in loading state_dict for MOMENTPipeline: ...


## 10.5 epoch별 "학습된 부분만" 체크포인트

전체 체크포인트는 Teacher 기준 **457 MB**입니다. 대부분은 사전학습 값 그대로인 몸통입니다. 그래서 9장 코드는 epoch마다 **LoRA + head만(약 1.2 MB)** 따로 저장합니다. 불러올 때는 사전학습 Base 위에 이것만 덮어씁니다.

In [7]:
import os
epochs_dir = ROOT / "outputs/manual/teacher/epochs"
for f in sorted(epochs_dir.glob("*.pt")):
    e = ch10.load_checkpoint(f)
    print(f"{f.name}: {os.path.getsize(f) / 1e6:.1f} MB, 텐서 {len(e['trainable_state_dict'])}개, validation loss {e['validation']['loss']:.4f}")

restored, n_part, n_missing = ch10.load_from_epoch(ROOT / "models/MOMENT-1-base", epochs_dir / f"epoch_{ckpt['epoch']:02d}.pt")
x = torch.randn(4, 1, 512)
with torch.no_grad():
    print("\n전체 체크포인트와 예측이 같은가?", torch.allclose(teacher(x_enc=x).logits, restored(x_enc=x).logits, atol=1e-6))

epoch_01.pt: 1.2 MB, 텐서 50개, validation loss 0.5684
epoch_02.pt: 1.2 MB, 텐서 50개, validation loss 0.2651
epoch_03.pt: 1.2 MB, 텐서 50개, validation loss 0.0176


Loading weights from local directory

전체 체크포인트와 예측이 같은가? True


## 10.6 체크포인트 사고 사례

| 사고 | 교훈 |
|---|---|
| 원하던 epoch 4 모델이 epoch 9, 10에 덮어쓰여 사라짐 | epoch별로 따로 저장한다 |
| LoRA B가 전부 0인 체크포인트를 20 epoch 학습 결과로 배포할 뻔함 | 저장된 값을 열어서 확인한다 |
| 체크포인트에 어떤 모델 폴더를 썼는지 기록이 없음 | `training_config`에 `model_dir`를 넣는다 (매뉴얼 코드는 기록함) |

## 10장 완성 코드: `code/ch10_checkpoint.py`

In [8]:
%run code/ch10_checkpoint.py

outputs/manual/teacher/teacher_lora_best.pt  (457.6 MB)
  epoch                  3
  model_state_dict       dict, 항목 164개, 숫자 113,771,906개 (455.1 MB)
  optimizer_state_dict   dict, 항목 2개
  validation_accuracy    1.0
  lora_config            dict, 항목 6개
  training_config        dict, 항목 9개

LoRA B 행렬 중 0이 아닌 것: 24/24  → 학습됨


Loading weights from local directory


Loading weights from local directory

epoch_03.pt: 학습된 텐서 50개만 덮어씀 (나머지 114개는 사전학습 값)
전체 체크포인트와 예측이 같은가? True


In [9]:
from IPython.display import Code
Code(filename=str(CODE / "ch10_checkpoint.py"), language="python")

"""10장 완성 코드: 체크포인트(.pt) 안을 들여다보고, 다시 모델로 불러온다.

- 전체 체크포인트(teacher_lora_best.pt)를 strict 모드로 불러오기
- epoch별로 저장한 "학습된 부분만" 체크포인트를 사전학습 모델 위에 덮어 불러오기
- LoRA B 행렬이 0인지 검사해 LoRA가 실제로 학습됐는지 확인하기

실행 (프로젝트 루트에서, 9장 결과 사용):
    python docs/manual/code/ch10_checkpoint.py
    python docs/manual/code/ch10_checkpoint.py --checkpoint outputs/checkpoints/teacher_lora_best.pt   # 옛날 모델
"""

import argparse
import warnings
from pathlib import Path

import torch

from ch07_moment import ROOT, load_moment
from ch08_lora import LORA_CONFIG, add_lora


def load_checkpoint(path):
    # weights_only=True: 텐서와 기본 자료형만 읽는다. 남이 만든 .pt 안의 악성 코드 실행을 막는다.
    return torch.load(path, map_location="cpu", weights_only=True)


def describe(path):
    ckpt = load_checkpoint(path)
    print(f"{Path(path).relative_to(ROOT)}  ({Path(path).stat().st_size / 1e6:.1f} MB)")
    for key, value in ckpt.items():
        if isinstance(value, dict):
            tensors = [v for v in value.values() if torch.is_tensor(v)]
            n = sum(t.numel() for t in tensors)
            print(f"  {key:22s} dict, 항목 {len(value)}개" + (f", 숫자 {n:,}개 ({n * 4 / 1e6:.1f} MB)" if tensors else ""))
        else:
            print(f"  {key:22s} {value}")
    return ckpt


def lora_b_report(state):
    """LoRA B는 0으로 시작한다. 학습이 됐다면 0이 아닌 값이 있어야 한다."""
    b = {k: v for k, v in state.items() if "lora_B" in k}
    nonzero = sum(int(v.abs().sum() > 0) for v in b.values())
    return nonzero, len(b)


def load_full(model_dir, path, teacher=True):
    """전체 state_dict를 strict=True로 불러온다. 이름 하나라도 안 맞으면 에러 → 조용한 실수를 막는다."""
    ckpt = load_checkpoint(path)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = load_moment(model_dir)
        if teacher:
            model = add_lora(model, ckpt.get("lora_config", LORA_CONFIG))
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    return model.eval()


def load_from_epoch(model_dir, path):
    """epoch별 파일(LoRA + head만)을 사전학습 Base 위에 덮어쓴다. 나머지는 사전학습 값 그대로."""
    part = load_checkpoint(path)["trainable_state_dict"]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = add_lora(load_moment(model_dir))
    result = model.load_state_dict(part, strict=False)
    # strict=False일 때는 결과를 직접 확인해야 한다: 모르는 이름(unexpected)이 있으면 안 된다
    assert not result.unexpected_keys, result.unexpected_keys
    missing_trainable = [k for k in result.missing_keys if "lora_" in k or k.startswith("head.")]
    assert not missing_trainable, missing_trainable
    return model.eval(), len(part), len(result.missing_keys)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", default="outputs/manual/teacher/teacher_lora_best.pt")
    parser.add_argument("--model-dir", default="models/MOMENT-1-base")
    args = parser.parse_args()

    ckpt = describe(ROOT / args.checkpoint)
    nonzero, total = lora_b_report(ckpt["model_state_dict"])
    print(f"\nLoRA B 행렬 중 0이 아닌 것: {nonzero}/{total}  → {'학습됨' if nonzero == total else 'LoRA가 학습되지 않음!'}")

    epochs_dir = (ROOT / args.checkpoint).parent / "epochs"
    if epochs_dir.exists():
        model_dir = ROOT / args.model_dir
        full = load_full(model_dir, ROOT / args.checkpoint)
        best_epoch = ckpt["epoch"]
        part_model, n_part, n_missing = load_from_epoch(model_dir, epochs_dir / f"epoch_{best_epoch:02d}.pt")
        torch.manual_seed(0)
        x = torch.randn(4, 1, 512)
        with torch.no_grad():
            same = torch.allclose(full(x_enc=x).logits, part_model(x_enc=x).logits, atol=1e-6)
        print(f"\nepoch_{best_epoch:02d}.pt: 학습된 텐서 {n_part}개만 덮어씀 (나머지 {n_missing}개는 사전학습 값)")
        print(f"전체 체크포인트와 예측이 같은가? {same}")


if __name__ == "__main__":
    main()